In [2]:
import re

# 1. Definimos um texto de teste
#texto_bruto = "Olá, mundo! Este é o nosso primeiro teste de tokenização."
texto_bruto = "token1 token2 token3 teste"
# 2. Usamos uma expressão regular (regex) para separar palavras e pontuações
# Essa regra diz: "corte o texto sempre que encontrar um espaço OU uma pontuação"
resultado_bruto = re.split(r'([,.:;?_!"()\']|--|\s)', texto_bruto)

# 3. Limpamos a lista removendo espaços vazios indesejados
tokens = [item for item in resultado_bruto if item.strip()]

print("--- RESULTADO DA TOKENIZAÇÃO ---")
print(f"Texto original: {texto_bruto}")
print(f"Quantidade de tokens: {len(tokens)}")
print(f"Lista de tokens: {tokens}")

--- RESULTADO DA TOKENIZAÇÃO ---
Texto original: token1 token2 token3 teste
Quantidade de tokens: 4
Lista de tokens: ['token1', 'token2', 'token3', 'teste']


In [3]:
# --- PASSO 2: Vocabulário e Tokens Especiais ---

# 1. Pegamos os tokens únicos e adicionamos os tokens especiais do GPT
tokens_unicos = sorted(set(tokens))
tokens_unicos.extend(["<|unk|>", "<|endoftext|>"])

# 2. Criamos o dicionário do vocabulário (Token -> ID)
vocab = {token: id_num for id_num, token in enumerate(tokens_unicos)}

# 3. Função para converter tokens em IDs lidando com palavras desconhecidas
def tokens_para_ids(lista_tokens, dicionario_vocab):
    ids = []
    for t in lista_tokens:
        # Se a palavra existir no vocabulário, pega o ID dela; senão, usa o ID de <|unk|>
        id_encontrado = dicionario_vocab.get(t, dicionario_vocab["<|unk|>"])
        ids.append(id_encontrado)
    return ids

# 4. Convertendo a frase original
token_ids = tokens_para_ids(tokens, vocab)

# 5. Testando com uma palavra nova ("carro") que NÃO está no texto inicial
frase_teste = ["token1", "carro", "teste"]
ids_teste = tokens_para_ids(frase_teste, vocab)

print(f"Tamanho do Vocabulário: {len(vocab)}")
print(f"Vocabulário: {vocab}\n")
print(f"Frase original em IDs: {token_ids}")
print(f"Teste com 'carro' (palavra desconhecida): {ids_teste}")

Tamanho do Vocabulário: 6
Vocabulário: {'teste': 0, 'token1': 1, 'token2': 2, 'token3': 3, '<|unk|>': 4, '<|endoftext|>': 5}

Frase original em IDs: [1, 2, 3, 0]
Teste com 'carro' (palavra desconhecida): [1, 4, 0]


In [5]:
# --- PASSO 3: Preparação das Sequências (Entradas e Alvos) ---

# Texto mais longo para podermos criar várias sequências
texto_treino = "isso é um segundo teste de controle e segurança do funcionamento de tokens"
tokens_treino = [item for item in re.split(r'([,.:;?_!"()\']|--|\s)', texto_treino) if item.strip()]

# Atualizamos nosso vocabulário para esse texto
vocab_treino = {token: id_num for id_num, token in enumerate(sorted(set(tokens_treino)))}
vocab_inverso = {id_num: token for token, id_num in vocab_treino.items()}
ids_treino = [vocab_treino[t] for t in tokens_treino]

# Tamanho do contexto: quantos tokens o modelo olha para tentar adivinhar o próximo
tamanho_contexto = 4

entradas = [] # x
alvos = []    # y

# Janela deslizante que percorre o texto criando os pares
for i in range(len(ids_treino) - tamanho_contexto):
    x = ids_treino[i : i + tamanho_contexto]
    y = ids_treino[i + 1 : i + tamanho_contexto + 1]
    entradas.append(x)
    alvos.append(y)

# Vamos inspecionar as duas primeiras amostras criadas
print(f"Total de pares (amostras) gerados: {len(entradas)}\n")

for i in range(2):
    txt_x = [vocab_inverso[idx] for idx in entradas[i]]
    txt_y = [vocab_inverso[idx] for idx in alvos[i]]
    print(f"Amostra {i+1}:")
    print(f"  Entrada (x) [IDs: {entradas[i]}]: {txt_x}")
    print(f"  Alvo    (y) [IDs: {alvos[i]}]: {txt_y}\n")

Total de pares (amostras) gerados: 9

Amostra 1:
  Entrada (x) [IDs: [5, 11, 10, 6]]: ['isso', 'é', 'um', 'segundo']
  Alvo    (y) [IDs: [11, 10, 6, 8]]: ['é', 'um', 'segundo', 'teste']

Amostra 2:
  Entrada (x) [IDs: [11, 10, 6, 8]]: ['é', 'um', 'segundo', 'teste']
  Alvo    (y) [IDs: [10, 6, 8, 1]]: ['um', 'segundo', 'teste', 'de']

